In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings("ignore")
print("✅ Imports done!")

df = pd.read_csv(
    r"C:\Users\Jyoti Hanagandi\OneDrive\Desktop\cyber-threat-detection\data\features.csv"
)

with open(
    r"C:\Users\Jyoti Hanagandi\OneDrive\Desktop\cyber-threat-detection\data\class_weights.json"
) as f:
    class_weights = json.load(f)

X = df.drop("label", axis=1)
y = df["label"]

print(f"Features shape: {X.shape}")
print(f"Class weights: {class_weights}")

# ── Three-way split: train / calibration / test ──
# Models train on X_train, get their confidence scores fixed on X_calib,
# and get evaluated on X_test — which stays completely untouched by
# training or calibration, so the reported accuracy stays honest.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_calib, y_train, y_calib = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)
print(f"Training set:     {X_train.shape}")
print(f"Calibration set:  {X_calib.shape}")
print(f"Test set:         {X_test.shape}")

print("Training Random Forest...")
print("This takes 3-5 minutes...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1, class_weight={0: class_weights["0"], 1: class_weights["1"]})
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
print(f"\n✅ Random Forest trained! Accuracy: {rf_acc*100:.2f}%")
print(classification_report(y_test, rf_pred, target_names=["Safe", "Malicious"]))

print("Training XGBoost...")
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, scale_pos_weight=scale_pos_weight, eval_metric="logloss", verbosity=0)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)
print(f"\n✅ XGBoost trained! Accuracy: {xgb_acc*100:.2f}%")
print(classification_report(y_test, xgb_pred, target_names=["Safe", "Malicious"]))

# ── Calibration — fixes the "98% confidence on everything" problem ──
# RF and XGBoost's raw predict_proba() is just tree-vote fractions, not a
# true probability, so it tends to be overconfident. FrozenEstimator marks
# the already-trained model as "don't refit me" — CalibratedClassifierCV
# then just learns a confidence-correction curve on the calibration set,
# without retraining the underlying RF/XGBoost model at all.
print("\nCalibrating models...")
calibrated_rf = CalibratedClassifierCV(FrozenEstimator(rf_model), method="sigmoid")
calibrated_rf.fit(X_calib, y_calib)

calibrated_xgb = CalibratedClassifierCV(FrozenEstimator(xgb_model), method="sigmoid")
calibrated_xgb.fit(X_calib, y_calib)

# Sanity check — calibrated accuracy should be very close to the raw
# model's accuracy (calibration adjusts confidence, not the verdict itself)
calibrated_rf_pred = calibrated_rf.predict(X_test)
calibrated_xgb_pred = calibrated_xgb.predict(X_test)
calibrated_rf_acc = accuracy_score(y_test, calibrated_rf_pred)
calibrated_xgb_acc = accuracy_score(y_test, calibrated_xgb_pred)
print(f"✅ Calibrated Random Forest accuracy: {calibrated_rf_acc*100:.2f}% (raw was {rf_acc*100:.2f}%)")
print(f"✅ Calibrated XGBoost accuracy:       {calibrated_xgb_acc*100:.2f}% (raw was {xgb_acc*100:.2f}%)")

import os
import joblib

os.makedirs("../models", exist_ok=True)

# Save the CALIBRATED models — these are what main.py will load.
# They still support .predict_proba(), so no backend changes are needed.
joblib.dump(calibrated_rf, "../models/random_forest.pkl")
joblib.dump(calibrated_xgb, "../models/xgboost.pkl")

print("✅ Both calibrated models saved!")
print(f"\nRandom Forest: {calibrated_rf_acc*100:.2f}%")
print(f"XGBoost:       {calibrated_xgb_acc*100:.2f}%")

RF_WEIGHT  = 0.55
XGB_WEIGHT = 0.45
rf_proba       = calibrated_rf.predict_proba(X_test)[:, 1]
xgb_proba      = calibrated_xgb.predict_proba(X_test)[:, 1]
ensemble_proba = RF_WEIGHT * rf_proba + XGB_WEIGHT * xgb_proba
ensemble_pred  = (ensemble_proba >= 0.5).astype(int)
ensemble_acc = accuracy_score(y_test, ensemble_pred)
print(f"Random Forest: {calibrated_rf_acc*100:.2f}%")
print(f"XGBoost:       {calibrated_xgb_acc*100:.2f}%")
print(f"Ensemble:      {ensemble_acc*100:.2f}%")
ensemble_config = {"rf_weight": RF_WEIGHT, "xgb_weight": XGB_WEIGHT, "rf_accuracy": calibrated_rf_acc, "xgb_accuracy": calibrated_xgb_acc, "ensemble_accuracy": ensemble_acc, "threshold": 0.5}
with open("../models/ensemble_config.json","w") as f:
    json.dump(ensemble_config, f, indent=2)

print("\n✅ STEP 4 + 5 COMPLETE! (Calibrated)")
print("Run step7_evaluation.ipynb next to regenerate metrics.json,")
print("the confusion matrix, and the ROC curve for the calibrated models.")

✅ Imports done!
Features shape: (641120, 34)
Class weights: {'0': 0.7488308260138292, '1': 1.5046986701965368}
Training set:     (410316, 34)
Calibration set:  (102580, 34)
Test set:         (128224, 34)
Training Random Forest...
This takes 3-5 minutes...

✅ Random Forest trained! Accuracy: 95.60%
              precision    recall  f1-score   support

        Safe       0.98      0.96      0.97     85616
   Malicious       0.92      0.95      0.94     42608

    accuracy                           0.96    128224
   macro avg       0.95      0.96      0.95    128224
weighted avg       0.96      0.96      0.96    128224

Training XGBoost...

✅ XGBoost trained! Accuracy: 94.43%
              precision    recall  f1-score   support

        Safe       0.97      0.95      0.96     85616
   Malicious       0.90      0.94      0.92     42608

    accuracy                           0.94    128224
   macro avg       0.93      0.94      0.94    128224
weighted avg       0.95      0.94      0.94  